# 03 — Establish the unchanged and filtering controls

## Question

How much accuracy and motion preservation can simple temporal processing provide under the same information and latency?

## Inputs

The validated bundle and development partition. Every method observes the same 64 samples at 25 Hz, approximately 2.52 seconds from first to last sample.

Use an explicit `STV2_CONFIG` JSON file and a unique run ID. The setup resolves relative artifact paths from the repository root. See the [execution guide](README.md), [development protocol](../../docs/studies/synthetic-training-v2/protocol.md) and [literature ledger](../../docs/studies/synthetic-training-v2/literature.md).

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Image, display

if not os.environ.get("STV2_CONFIG"):
    raise RuntimeError("Set STV2_CONFIG to an explicit study JSON configuration before execution.")
config_path = Path(os.environ["STV2_CONFIG"]).expanduser().resolve()
if not config_path.is_file():
    raise FileNotFoundError(f"Study configuration does not exist: {config_path}")
search_root = Path(os.environ.get("GAVD6_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next((path for path in (search_root, *search_root.parents)
                     if (path / "src/gavd6_sjepa").is_dir() and (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run inside the repository or set GAVD6_ROOT to its root.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ["STV2_CONFIG"] = str(config_path)
from gavd6_sjepa.research_directions.synthetic_training_v2.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training_v2.workflow import run_stage

cfg = RunConfig.load(os.environ["STV2_CONFIG"])
display({"run_id": cfg.run_id, "mode": cfg.mode, "device": cfg.device,
         "artifact_root": str(cfg.root), "confirmation": "closed"})
if cfg.mode == "fixture":
    print("CPU software fixture: method ordering is not empirical evidence.")

## Computation

Save unchanged predictions and the predeclared filter strengths 0, 1 and 2 frames for each configured seed. Use input coordinates, confidence, observed flags and timestamps only; preserve target records separately for scoring.

`run_stage` implements the computation in the study modules. It checks prerequisite receipts and returns the saved result on an unchanged rerun; a changed configuration or code identity requires a new run ID.

In [ ]:
information = run_stage(cfg, "information", repo_root=PROJECT_ROOT)
display(information)

## Outputs and checks

Check `predictions/unchanged-*.npz`, `predictions/filter*-*.npz` and their record sidecars. Predictions retain physical timestamps and evaluation masks so subsequent tables can be rebuilt. Cadence remains unsupported when the window lacks adequate cycles.

Stage receipts under `receipts/` record elapsed time and hashes of produced artifacts. Inspect the saved files for full diagnostics; the display above is deliberately brief.

## Interpretation

Smoother trajectories can erase real amplitude or shift events. SmoothNet and SynSP establish temporal refinement as prior work; coordinate accuracy alone cannot establish that gait motion survived. These are offline observed-window controls, not forecasting.

A completed fixture checks software behavior. Scientific gates use `pass`, `fail` or `insufficient_evidence`; fixture success cannot make a scientific gate pass.

## Next gate

Proceed to [04 — coordinate versus JEPA](04_coordinate_vs_jepa.ipynb). Keep all predeclared filtering strengths in the tradeoff plot and select nothing using confirmation data.